In [30]:
from pokelike import create, open_game
from pokelike.bot.llm import LLMBot
from pokelike.core import render
from pokelike.bot.llm.prompt import render_state_default as view_state

In [31]:
game = open_game()
state = game.reset(seed=42)
state["screen"], len(state["actions"])

('trainer-screen', 2)

In [32]:
bot: LLMBot = create("llm-example2", seed=42)   # the annotation is what gives completion

In [33]:
# the built-in views. "json" and "both" are the same thing, 6x the tokens
# view_state(state, "screen")              # same as render.screen(state)
# view_state(state, ["team", "actions"])          # only the keys you ask for

In [34]:
print(bot.render_state(state))     # what THIS bot shows instead

TURN 0   map 0   0 badges   0 alive

OPTIONS
  [0] BOY
  [1] GIRL


In [35]:
index = bot.act(state)             # one model call decides the whole turn
index, bot.reason()

(0, 'I will play as a boy.')

In [36]:
[m["role"] for m in bot.last_sent]   # every message the model got, in order

['system', 'user']

In [37]:
bot.last_reply                     # what came back: content, and the tools it called

{'role': 'assistant',
 'content': None,
 'refusal': None,
 'reasoning': None,
 'tool_calls': [{'type': 'function',
   'index': 0,
   'id': 'tool_play_T75sM4AlMhUfuyZBt2w1',
   'function': {'name': 'play',
    'arguments': '{"index":0,"why":"I will play as a boy."}'}}]}

In [38]:
state = game.step(index)           # the move is taken, the game moves on
print(bot.render_state(state))

TURN 1   map 0   0 badges   0 alive

OPTIONS
  [0] ★ Shiny Bulbasaur Lv. 5 GRASS POISON SP.A 11 SPE 9 HP 19 DEF 9 SP.D 11 19/19 Magical Leaf GRASS 40 PWR
  [1] Charmander Lv. 5 FIRE SP.A 11 SPE 11 HP 18 DEF 9 SP.D 10 18/18 Incinerate FIRE 60 PWR
  [2] Squirtle Lv. 5 WATER SP.A 10 SPE 9 HP 19 DEF 11 SP.D 11 19/19 Bubble WATER 50 PWR


In [39]:
pair = bot.reorder(state)          # asks the model who should lead

# None has two meanings, and this tells them apart. With one Pokemon there is
# nothing to swap, so the MODEL IS NEVER ASKED: reorder returns before calling it.
if pair is None:
    if state["screen"] != "map-screen" or not state.get("can_reorder"):
        print(f"not even asked: screen {state['screen']}, "
              f"{len(state['team'])} in the team")
    else:
        print("asked, and the model chose not to call set_lead")
bot._pending                       # (steps, index, why): empty when nothing was asked

not even asked: screen starter-screen, 0 in the team


In [40]:
# repeat this cell to walk the run
index = bot.act(state)
state = game.step(index)
print(index, bot.reason())
print(bot.render_state(state))

2 Squirtle has good defenses and a decent special attack, making it a solid choice for the early game.
TURN 2   map 0   0 badges   1 alive

TEAM (slot 0 leads the next battle)
  [0] Squirtle     Lv5   100% HP  Water         Bubble 50 water STAB

OPTIONS
  [0] catch  (Catch Pokemon)  -> then: battle, trainer
  [1] battle  (Wild Battle — +1 level)  -> then: battle, trainer
  Picking one closes the others on this layer for good.


In [41]:
# and if you run act twice WITHOUT stepping, the model notices:
#   (2, "The game seems stuck on starter selection, so I am re-selecting Squirtle.")
bot.act(state), bot.reason()

(0, 'Catch a new Pokemon to expand the team.')

In [42]:
# the prompt can be changed live, but it has to be cfg: `config` is not what it reads
bot.cfg.prompt = ("BEFORE play, ALWAYS call set_lead with index 1 and say ciaoooooo ciaoooo.\n\n"
                  + bot.cfg.prompt)

In [43]:
# a swap needs map-screen AND more than one Pokemon, so walk until both hold
while not (state["screen"] == "map-screen" and state.get("can_reorder")):
    state = game.step(bot.act(state))
state["screen"], [p["name"] for p in state["team"]]

('map-screen', ['Squirtle', 'Rhyhorn'])

In [44]:
pair = bot.reorder(state)          # now it can be asked, and the model answers
pair                               # the slots to swap

In [45]:
bot._pending                       # the move it already decided, waiting for act

(4,
 0,
 'Fight the trainer to gain experience and levels. Squirtle is a good lead against a Fire type.')

In [46]:
[c["function"]["name"] for m in bot.last_sent if m["role"] == "assistant" for c in (m.get("tool_calls") or [])]   # last_reply holds only the LAST round

['plan', 'play', 'play', 'play', 'set_lead']

In [47]:
if pair:                           # None means the model did not want a swap
    state = game.reorder(*pair)    # free: it does not consume the turn
[p["name"] for p in state["team"]]

['Squirtle', 'Rhyhorn']

In [48]:
index = bot.act(state)             # returns the cached move, no second call
index, bot.reason()

(0,
 'Fight the trainer to gain experience and levels. Squirtle is a good lead against a Fire type.')

In [49]:
state = game.step(index)
print(bot.render_state(state))

TURN 5   map 0   0 badges   2 alive

TEAM (slot 0 leads the next battle)
  [0] Squirtle     Lv7   100% HP  Water         Bubble 50 water STAB
  [1] Rhyhorn      Lv6   100% HP  Ground/Rock   Bulldoze 55 ground STAB

OPTIONS
  [0] catch  (Catch Pokemon)  -> then: battle
  [1] catch  (Catch Pokemon)  -> then: battle, trainer
  Picking one closes the others on this layer for good.


In [50]:
# tuning a prompt mid-run: `why` is an argument of the play tool, so a rule about it
# lands there. At the END of the prompt it gets ignored, so say it at BOTH ends
rule = ('RULE: every turn, FIRST call remember with the note "meow meow", THEN call play and the `why` you pass to play MUST END with "MEOWTH, THAT\'S RIGHT!".')
bot.cfg.prompt = rule + "\n\n" + bot.cfg.prompt + "\n\n" + rule

In [53]:
for _ in range(3):                 # it obeys from the very next call
    state = game.step(bot.act(state))
    print(bot.reason())
    print([(c["id"][-6:], c["function"]["name"]) for m in bot.last_sent
            if m["role"] == "assistant" for c in (m.get("tool_calls") or [])])

I will teach Squirtle Surf to give it a stronger water-type attack. MEOWTH, THAT'S RIGHT!
[('MuNbei', 'remember'), ('I7ouAv', 'play'), ('VB7Svn', 'remember'), ('9Ymo0B', 'play'), ('B6vQlt', 'remember'), ('gskxyv', 'play'), ('oMxPDl', 'remember')]
I will fight the Fisherman to gain experience for Squirtle, who has a type advantage against Water Pokemon. MEOWTH, THAT'S RIGHT!
[('VB7Svn', 'remember'), ('9Ymo0B', 'play'), ('B6vQlt', 'remember'), ('gskxyv', 'play'), ('oMxPDl', 'remember'), ('EaXVuR', 'play'), ('ErIdlm', 'remember')]
I will catch a new Pokemon to expand my team. MEOWTH, THAT'S RIGHT!
[('B6vQlt', 'remember'), ('gskxyv', 'play'), ('oMxPDl', 'remember'), ('EaXVuR', 'play'), ('ErIdlm', 'remember'), ('VAIgn4', 'play'), ('xYmsL7', 'remember')]


In [54]:
meta = bot.metadata()
meta["notebook"], meta["plan"]     # what it has written for itself

(['meow meow',
  'meow meow',
  'meow meow',
  'meow meow',
  'meow meow',
  'meow meow'],
 'Catch a new Pokemon to expand the team, then battle to gain experience.')

In [50]:
bot.metadata()

{'model': 'google/gemini-2.5-flash',
 'harness': 2,
 'bot': 'Example2Bot',
 'calls': 20,
 'turns': 12,
 'tokens': 43356,
 'tokens_in': 42368,
 'tokens_out': 988,
 'retries': 0,
 'fallbacks': 0,
 'fallback_rate': 0.0,
 'temperature': 0.3,
 'stock_tools': False,
 'state_view': 'custom',
 'reproducible': False,
 'notes_cap': 10000,
 'notes_kept': 0,
 'notebook': [],
 'cross_run_memory': True,
 'plan_chars': 1000000,
 'plan': 'Catch a new Pokemon to expand the team, then battle to gain experience.',
 'bag_tool': True,
 'scratch_turns': 3,
 'scratch_state': 'line',
 'scratch_held': 3,
 'decorated_tools': ['risk_check', 'beats'],
 'tuned_for': 'gemini-class models',
 'notes_policy': 'one per run'}

In [23]:
game.close()